# Notebook 02 — ETL Pipeline & Data Cleaning

> All raw files are read-only. Every transformation is logged. Output to `data/processed/`.

In [ ]:
import pandas as pd
import numpy as np

matches = pd.read_csv('../data/raw/matches.csv')
deliveries = pd.read_csv('../data/raw/deliveries.csv')
print(f'[LOAD] matches: {matches.shape}, deliveries: {deliveries.shape}')

## T1 — Season Label Standardisation

In [ ]:
season_map = {'2007/08':'2008','2009/10':'2010','2020/21':'2021'}
matches['season'] = matches['season'].replace(season_map).astype(int)
print('Seasons after fix:', sorted(matches['season'].unique()))

## T2 — Date Parsing

In [ ]:
matches['date'] = pd.to_datetime(matches['date'])
print(matches['date'].dtype)

## T3 — City Imputation from Venue

In [ ]:
venue_city = {
    'Dubai International Cricket Stadium':'Dubai',
    'Sheikh Zayed Stadium':'Abu Dhabi',
    'Sharjah Cricket Stadium':'Sharjah',
    'Zayed Cricket Stadium, Abu Dhabi':'Abu Dhabi',
}
matches['city'] = matches.apply(
    lambda r: venue_city.get(r['venue'], r['city']) if pd.isna(r['city']) else r['city'], axis=1
)
print('Remaining null cities:', matches['city'].isna().sum())

## T4 — Team Name Standardisation (Franchise Renames)

In [ ]:
team_rename = {
    'Delhi Daredevils':'Delhi Capitals',
    'Kings XI Punjab':'Punjab Kings',
    'Rising Pune Supergiant':'Rising Pune Supergiants',
    'Royal Challengers Bangalore':'Royal Challengers Bengaluru',
}
for col in ['team1','team2','toss_winner','winner']:
    matches[col] = matches[col].replace(team_rename)
deliveries['batting_team'] = deliveries['batting_team'].replace(team_rename)
deliveries['bowling_team'] = deliveries['bowling_team'].replace(team_rename)
print('Unique teams after standardisation:', matches['team1'].nunique())

## T5–T9 — Derived Columns & Semantic Fills

In [ ]:
matches['toss_win_match'] = (matches['toss_winner'] == matches['winner']).astype(int)
matches['method'] = matches['method'].fillna('Normal')
deliveries['extras_type'] = deliveries['extras_type'].fillna('none')
deliveries['player_dismissed'] = deliveries['player_dismissed'].fillna('not_out')
deliveries['dismissal_kind'] = deliveries['dismissal_kind'].fillna('not_out')
deliveries['fielder'] = deliveries['fielder'].fillna('none')

def get_phase(over):
    if over < 6: return 'Powerplay'
    elif over < 15: return 'Middle'
    else: return 'Death'

deliveries['phase'] = deliveries['over'].apply(get_phase)
super_over_matches = matches[matches['super_over']=='Y']['id'].tolist()
deliveries['is_super_over_match'] = deliveries['match_id'].isin(super_over_matches)
print('Phase distribution:')
print(deliveries['phase'].value_counts())

## Validation & Save

In [ ]:
print('Cleaned matches nulls:')
print(matches[['id','season','city','winner','toss_decision']].isnull().sum())
print('\nCleaned deliveries nulls:')
print(deliveries[['batting_team','bowler','phase','is_wicket']].isnull().sum())

matches.to_csv('../data/processed/matches_clean.csv', index=False)
deliveries.to_csv('../data/processed/deliveries_clean.csv', index=False)
print('\n[SAVE] Written to data/processed/')